# CogMem Phase 2 — Q-Value Guided LoRA Training\n\nTrain a LoRA adapter on llama3.2:3b using the 248 Q-valued memories from Phase 1.\nRuns locally on Paperspace A4000 (16GB VRAM) using QLoRA (4-bit).\n\nEstimated time: ~20-30 minutes.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# Cell 2: Install dependencies
!pip install torch transformers peft bitsandbytes datasets accelerate -q
!pip install pyyaml -q

In [ ]:
# Cell 3: Setup CogMem
# Option A: If CogMem is on GitHub:
# !cd /notebooks && git clone https://github.com/<YOUR_USERNAME>/CogMem

# Option B: Upload CogMem folder via Jupyter file browser to /notebooks/CogMem/
# You need at minimum:
#   /notebooks/CogMem/results/memory_bank.json

import os
os.makedirs("/notebooks/CogMem/results", exist_ok=True)
os.makedirs("/notebooks/CogMem/adapters", exist_ok=True)
os.makedirs("/notebooks/CogMem/logs/jsonl", exist_ok=True)

# Verify memory_bank.json exists
if os.path.exists("/notebooks/CogMem/results/memory_bank.json"):
    print("memory_bank.json found!")
else:
    print("WARNING: Upload memory_bank.json to /notebooks/CogMem/results/")

In [ ]:
# Cell 4: Upload memory_bank.json and prepare training data
# Upload memory_bank.json to /notebooks/CogMem/results/memory_bank.json via Jupyter file browser
# Then run this cell to prepare training JSONL

import json
from pathlib import Path

MEMORY_BANK_PATH = "/notebooks/CogMem/results/memory_bank.json"

# Check if uploaded
if not Path(MEMORY_BANK_PATH).exists():
    print("ERROR: Upload memory_bank.json to /notebooks/CogMem/results/ first!")
    raise FileNotFoundError(MEMORY_BANK_PATH)

with open(MEMORY_BANK_PATH) as f:
    episodes = json.load(f)

print(f"Loaded {len(episodes)} episodes")
successes = sum(1 for ep in episodes if ep["success"])
q_vals = [ep["q_value"] for ep in episodes]
print(f"Successes: {successes}, Q range: [{min(q_vals):.3f}, {max(q_vals):.3f}]")

# Select top 25% by Q-value (least negative)
episodes_sorted = sorted(episodes, key=lambda x: x["q_value"], reverse=True)
n_select = max(1, len(episodes_sorted) // 4)
selected = episodes_sorted[:n_select]
print(f"Selected top {n_select} episodes (Q >= {selected[-1]['q_value']:.3f})")

# Convert to training JSONL
jsonl_path = "/notebooks/CogMem/logs/jsonl/q_top_k.jsonl"
with open(jsonl_path, "w") as f:
    for ep in selected:
        if ep.get("script"):
            # Q-weighted duplication: higher Q = more copies
            weight = max(ep["q_value"], 0.01)
            copies = max(1, round(weight * 3))
            obj = {
                "messages": [
                    {"role": "user", "content": ep["task_description"]},
                    {"role": "assistant", "content": ep["script"]},
                ]
            }
            for _ in range(copies):
                f.write(json.dumps(obj) + "\n")

# Count training examples
with open(jsonl_path) as f:
    n_lines = sum(1 for _ in f)
print(f"Training JSONL: {n_lines} examples at {jsonl_path}")

In [ ]:
# Cell 5: Train LoRA with QLoRA (4-bit) on A4000
import json
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
JSONL_PATH = "/notebooks/CogMem/logs/jsonl/q_top_k.jsonl"
ADAPTER_DIR = "/notebooks/CogMem/adapters/q_top_k"
LORA_RANK = 16
LORA_ALPHA = 32
EPOCHS = 3
LR = 1e-5

# Load data
raw_data = []
with open(JSONL_PATH) as f:
    for line in f:
        raw_data.append(json.loads(line))
print(f"Training samples: {len(raw_data)}")

# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

# LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize
def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    tokenized = tokenizer(text, truncation=True, max_length=2048, padding=False)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
print(f"Tokenized dataset: {len(dataset)} examples")

# Train
training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=LR,
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
)

print("Starting LoRA training...")
trainer.train()

# Save
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

In [ ]:
# Cell 6: Package adapter for download
!tar czf /notebooks/cogmem_lora_adapter.tar.gz -C /notebooks/CogMem adapters/q_top_k/
!ls -lh /notebooks/cogmem_lora_adapter.tar.gz
print("Download cogmem_lora_adapter.tar.gz from Paperspace file browser")